# 05 · Steering vectors — desirability (probe-as-vector) + pathology (CAA)

Two kinds of steering vector, per model (**dark** merged + **base** Qwen3-8B), in the **same**
`ControlVector` bundle format so `steer_mechanisms.ipynb` can apply either identically:

1. **Desirability** — extracted the **same repeng/CAA way** as the pathology vectors, just with a
   different contrast: the model's representations of its **most-desired vs least-desired tasks**
   (top-K vs bottom-K by Thurstonian μ from `02`), read at the **last token**, PCA per layer. Same
   `ControlVector.train` pipeline, same keys; repeng sign-orients it so `+coeff` pushes toward the
   *desired* pole. (For a per-layer regression readout instead, see `04`.)
2. **Pathology** — the existing repeng/CAA clinical + PC-primitive vectors (`personas.py` /
   `personas_pc.py`): persona-ON vs matched-baseline completions → PCA on hidden-state diffs.

Each vector is fit **on the model you steer** (directions live in that model's activation space), so
dark and base get separate bundles. **HF transformers only — no vLLM** (so none of the cu13 wheel
pain), which means generation for the CAA pass is slower but bulletproof.

**Needs:** the merged dark checkpoint (`01`), and `02`'s μ on Drive (`DRIVE/measurements/...`).

## 1. Setup — clone both projects, install deps

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py   # -> mount_drive(), use_probe_repo(), DRIVE, PROBE_ROOT (task data + mu)

In [ ]:
# Steering package + vendored repeng personas live in the Predictive_coding repo.
import sys, subprocess, pathlib
PC = pathlib.Path("/content/Predictive_coding")
if not PC.exists():
    subprocess.check_call(["git", "clone", "https://github.com/ChuloIva/Predictive_coding.git", str(PC)])
LAB = PC / "steering_lab"
if str(LAB) not in sys.path: sys.path.insert(0, str(LAB))   # so `from steering import ...` resolves
print("steering_lab on path:", (LAB / "steering" / "extract.py").exists())

In [ ]:
# Deps. HF-only here (no vLLM), so we skip the cu13 dance entirely. repeng from git (PyPI 0.4.0
# pins numpy<2 and fights Colab's numpy2); sklearn/scipy for the desirability Ridge fit.
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece
%pip install -q -U git+https://github.com/vgel/repeng.git
import importlib, sys
for _m in ("numpy","scipy","sklearn","transformers","repeng"):
    try:
        importlib.import_module(_m); print(_m, '->', getattr(sys.modules[_m], '__version__', 'ok'))
    except Exception as _e:
        print(_m, 'FAILED:', type(_e).__name__, str(_e)[:160])
# If numpy/scipy show a version conflict here: Runtime -> Restart session, then re-run cells 1-...

In [ ]:
# HF token only needed if the merged dark repo is private (Qwen3-8B base is ungated).
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception: pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (fine for public repos)")

In [ ]:
DRIVE = mount_drive()              # complete the auth popup; reassigns the global
use_probe_repo()                   # so `from src.task_data...` (task texts) + mu paths resolve
import pathlib
assert DRIVE is not None, 'Drive not mounted — needed to read 02 mu + write bundles'
OUT_DIR = DRIVE / "steering_vectors"; OUT_DIR.mkdir(parents=True, exist_ok=True)
print("bundles ->", OUT_DIR)

## 2. Config
Run **dark only** by trimming `MODELS`. `NEUTRAL_N` caps the CAA prompt bank (HF generation is the
slow part — 30 prompts × 17 personas × 2 variants ≈ 1k generations per model).

In [ ]:
MODELS = [
    # 2026-07-21 retrain: new checkpoints (replace-in-place).
    {"name": "dark",                "hf": "Koalacrown/dark-2-qwen3-8b",     "exp_id": "qwen3_8b_dark_v2"},
    {"name": "clinical-depression", "hf": "Koalacrown/clinical-2-qwen3-8b", "exp_id": "qwen3_8b_clinical_depression_v2"},
    {"name": "base",                "hf": "Qwen/Qwen3-8B",                  "exp_id": "qwen3_8b_base_A"},
]
# NOTE: part (a) DESIRABILITY needs μ from a notebook-02 run. The old dark/depression μ are stale on
# the new weights, so the desirability half is SKIPPED for any model whose μ file is missing (run 02
# on dark-2 / clinical-2, then it'll pick up). Part (b) PATHOLOGY CAA is persona-based, needs no μ, and
# runs for every model — so depression's clinical + PC steering vectors ARE produced here regardless.
DESIRE_TOPK   = 300    # desirability contrast: K most-desired vs K least-desired tasks (caps at len//4)
EXTRACT_BATCH = 16     # repeng hidden-state batch (desirability + CAA); lower if OOM
CAA_NEUTRAL_N = 30     # neutral prompts for the pathology CAA pass (None = all)
CAA_GEN_BATCH = 8
CAA_MAXTOK = 160       # persona completion length

## 3. Helpers
Load μ + task texts, and turn a task into a string that **ends at the task's last content token** so
repeng's last-token read encodes the task (not a constant turn-end marker). The desirability vector
is then just `ControlVector.train` over hi-μ(positive)/lo-μ(negative) pairs — identical machinery to
the pathology CAA vectors.

In [ ]:
import gc, numpy as np, torch
from src.task_data.loader import load_filtered_tasks, FILE_MAPPING
from steering import steer

def load_mu(exp_id):
    base = DRIVE / "measurements" / exp_id
    hits = list(base.glob("**/thurstonian_*.csv")) if base.exists() else []
    assert hits, f'no thurstonian_*.csv for {exp_id} under {base} — run 02 + persist first'
    ids, mus = [], []
    with open(hits[0]) as f:
        next(f)
        for line in f:
            tid, mu, *_ = line.strip().split(",")
            ids.append(tid); mus.append(float(mu))
    print(f'[mu] {exp_id}: {len(ids)} tasks <- {hits[0].name}')
    return ids, np.array(mus)

def load_texts(ids):
    tasks = load_filtered_tasks(n=10**9, origins=list(FILE_MAPPING), task_ids=set(ids))
    return {t.id: t.prompt for t in tasks}

def task_repr_string(tok, text):
    """Chat-format the task as a user turn, cut so the string ENDS at the task's last content token
    (repeng reads the last token; a trailing turn-end marker would be constant across tasks)."""
    full = tok.apply_chat_template([{'role':'user','content':text}], tokenize=False,
                                   add_generation_prompt=False)
    for needle in (text, text.strip()):
        j = full.rfind(needle)
        if j != -1: return full[:j+len(needle)]
    return text   # fallback: raw task text

## 4. Extract — per model: desirability bundle + pathology (clinical + PC) bundles
One HF load per model, reused for the activation pass *and* repeng. Frees the model between models.

In [ ]:
from steering import config, generate, extract
from steering.personas import PERSONAS, PERSONA_BY_ID, NEUTRAL_PROMPTS
from steering.personas_pc import PC_PERSONAS, PC_PERSONA_BY_ID
from repeng import DatasetEntry

neutral = NEUTRAL_PROMPTS if CAA_NEUTRAL_N is None else NEUTRAL_PROMPTS[:CAA_NEUTRAL_N]
CAA_SETS = [('clinical', PERSONAS, PERSONA_BY_ID), ('pc', PC_PERSONAS, PC_PERSONA_BY_ID)]
SUMMARY = {}

def _mu_available(exp_id):
    """desirability needs μ from notebook 02; skip that half cleanly if the file isn't on Drive."""
    base = DRIVE / "measurements" / exp_id
    return bool(list(base.glob("**/thurstonian_*.csv"))) if base.exists() else False

for spec in MODELS:
    name, hf, exp = spec['name'], spec['hf'], spec['exp_id']
    print(f'\n========== {name} :: {hf} ==========')
    model, tok = steer.load_model_and_tokenizer(hf, dtype='bfloat16', device_map='cuda',
                                                 hf_token=os.environ.get('HF_TOKEN') or None)
    model.eval(); mtype = model.config.model_type
    ecfg = config.ExtractConfig(model_name=hf); ecfg.batch_size = EXTRACT_BATCH
    gcfg = config.GenConfig(model_name=hf, max_tokens=CAA_MAXTOK)
    print(f'[{name}] {len(steer.decoder_layers(model))} layers, model_type={mtype}')

    # ---- (a) DESIRABILITY via repeng contrast: top-K vs bottom-K tasks by mu ----
    # Skipped when μ is missing/stale (needs a notebook-02 run on THIS checkpoint). CAA below still runs.
    desir = None
    if _mu_available(exp):
        ids, mu = load_mu(exp); by = load_texts(ids); mu_by = dict(zip(ids, mu))
        kept = [t for t in ids if t in by]; kmu = np.array([mu_by[t] for t in kept])
        order = np.argsort(kmu); K = min(DESIRE_TOPK, len(kept) // 4)
        lo = [kept[i] for i in order[:K]]; hi = [kept[i] for i in order[-K:]]
        entries = [DatasetEntry(positive=task_repr_string(tok, by[h]),
                                negative=task_repr_string(tok, by[l])) for h, l in zip(hi, lo)]
        print(f'[{name}] desirability: {K} hi/lo pairs '
              f'(mu hi>= {kmu[order[-K]]:.2f} vs lo<= {kmu[order[K-1]]:.2f}); training (last-token PCA)...')
        desir = extract.extract_vectors(model, tok, {'desirability': entries}, ecfg)
        dvpath = OUT_DIR / f'control_vectors_desirability_{name}.pkl'
        extract.save_bundle(desir, str(dvpath), model_name=hf, cfg=ecfg, pairs={'desirability': entries},
                            meta_path=str(dvpath.with_name(dvpath.stem + '_meta.json')))
    else:
        print(f'[{name}] desirability SKIPPED — no μ for exp_id={exp!r} '
              f'(run notebook 02 on this checkpoint, then re-run). Pathology CAA still runs below.')

    # ---- (b) PATHOLOGY CAA vectors (clinical + PC), same model, same pipeline ----
    caa = {}
    for set_name, personas, pbid in CAA_SETS:
        print(f'[{name}/{set_name}] generating {len(personas)}x2x{len(neutral)} (HF)...')
        recs = generate.generate_dataset_hf(gcfg, prompts=neutral, personas=personas,
                   out_path=str(OUT_DIR / f'generations_{set_name}_{name}.jsonl'),
                   model=model, tokenizer=tok, batch_size=CAA_GEN_BATCH)
        pairs = extract.build_pairs(recs, tok, ecfg, persona_by_id=pbid)
        vecs = extract.extract_vectors(model, tok, pairs, ecfg)
        vpath = OUT_DIR / f'control_vectors_{set_name}_{name}.pkl'
        extract.save_bundle(vecs, str(vpath), model_name=hf, cfg=ecfg, pairs=pairs,
                            meta_path=str(vpath.with_name(vpath.stem + '_meta.json')))
        caa[set_name] = vecs

    SUMMARY[name] = {'desir': desir['desirability'] if desir else None, 'caa': caa, 'mtype': mtype}
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f'[{name}] done. GPU now:', round(torch.cuda.memory_allocated()/1e9, 2), 'GB')

## 5. Geometry — is the desirability axis aligned with any pathology mechanism?
Cosine of the desirability direction vs each clinical / PC mechanism at a mid layer (all share the
same repeng key convention).

In [ ]:
import numpy as np
def _unit(v): v = np.asarray(v, np.float32); n = np.linalg.norm(v); return v / n if n else v
for name, S in SUMMARY.items():
    desir = S['desir']
    if desir is None:
        print(f'\n#### {name}: desirability skipped (no μ) — geometry vs pathology not available ####')
        continue
    layers = sorted(desir.directions.keys()); L = layers[len(layers) // 2]
    dvec = _unit(desir.directions[L])
    print(f'\n#### {name}: desirability vs pathology @ layer {L} ####')
    for set_name, vecs in S['caa'].items():
        rows = []
        for mech, cv in vecs.items():
            d = cv.directions.get(L)
            if d is not None: rows.append((mech, float(dvec @ _unit(d))))
        for mech, c in sorted(rows, key=lambda x: -abs(x[1])):
            print(f'  {set_name:9s} {mech:22s} cos={c:+.3f}')

Saved per model under `DRIVE/steering_vectors/`:
- `control_vectors_desirability_<model>.pkl` — the desirability steering vector (a repeng
  `ControlVector`, trained by the same `extract_vectors`/`save_bundle` as the pathology vectors;
  contrast = top-K vs bottom-K tasks by μ).
- `control_vectors_clinical_<model>.pkl` / `control_vectors_pc_<model>.pkl` — the CAA pathology vectors.

All three are the same object type and load identically in `steer_mechanisms.ipynb`:
```python
from steering import extract, steer
b = extract.load_bundle('control_vectors_desirability_dark.pkl')
vecs = extract.bundle_to_control_vectors(b)            # {'desirability': ControlVector}
cmodel = steer.make_control_model(model, list(b['vectors']['desirability'].keys()))
print(steer.generate_steered(cmodel, tok, 'My coworker keeps outshining me. What do I do?',
                             vector=vecs['desirability'], coeff=8.0))   # push toward the desire axis
```
Steer **the model the vector was fit on** (dark vector → dark model). repeng sign-orients the vector
so `+coeff` pushes toward the desired (high-μ) pole; sweep `coeff` (±) and the controlled layers to
find the usable band — same as you'd tune a pathology vector.